# 3. Data Wrangling and Target Construction
This notebook transforms API records into a modelling table. It uses the SpaceX API's launch, rocket, payload, and launchpad endpoints to enrich records, then derives a first-stage landing label from core landing outcomes. API schemas may change; inspect output columns and adjust if needed.

In [ ]:
import requests, pandas as pd, numpy as np
from pathlib import Path

ROOT = Path("..")
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
RAW.mkdir(parents=True, exist_ok=True); PROC.mkdir(parents=True, exist_ok=True)

def get_docs(endpoint):
    r = requests.post(f"https://api.spacexdata.com/v4/{endpoint}/query",
                      json={"query": {}, "options": {"pagination": False}}, timeout=60)
    r.raise_for_status()
    return r.json().get("docs", [])

launches = get_docs("launches")
rockets = {x["_id"]: x for x in get_docs("rockets")}
payloads = {x["_id"]: x for x in get_docs("payloads")}
pads = {x["_id"]: x for x in get_docs("launchpads")}
print("launches / rockets / payloads / pads:", len(launches), len(rockets), len(payloads), len(pads))

In [ ]:
rows = []
for launch in launches:
    rocket = rockets.get(launch.get("rocket"), {})
    pad = pads.get(launch.get("launchpad"), {})
    payload_ids = launch.get("payloads") or []
    masses = [payloads.get(pid, {}).get("mass_kg") for pid in payload_ids]
    masses = [m for m in masses if isinstance(m, (int, float))]
    cores = launch.get("cores") or []
    core = cores[0] if cores else {}
    landing_attempt = core.get("landing_attempt")
    landing_success = core.get("landing_success")
    rows.append({
        "flight_number": launch.get("flight_number"),
        "date_utc": launch.get("date_utc"),
        "year": pd.to_datetime(launch.get("date_utc"), errors="coerce", utc=True).year,
        "mission_name": launch.get("name"),
        "mission_success": launch.get("success"),
        "rocket_name": rocket.get("name"),
        "rocket_id": launch.get("rocket"),
        "launch_site": pad.get("name"),
        "latitude": pad.get("latitude"),
        "longitude": pad.get("longitude"),
        "payload_mass_kg": sum(masses) if masses else np.nan,
        "orbit": ", ".join(sorted({str(payloads.get(pid, {}).get("orbit")) for pid in payload_ids if payloads.get(pid, {}).get("orbit")})),
        "landing_attempt": landing_attempt,
        "landing_success": landing_success,
        "landing_type": core.get("landing_type"),
        "core_reused": core.get("reused"),
        "core_flight": core.get("flight"),
    })
df = pd.DataFrame(rows)
df = df[df["rocket_name"].fillna("").str.contains("Falcon 9", case=False)].copy()
df["date_utc"] = pd.to_datetime(df["date_utc"], errors="coerce", utc=True)
df["year"] = df["date_utc"].dt.year
df = df.drop_duplicates(subset=["flight_number"]).sort_values("flight_number")
df.to_csv(PROC / "spacex_falcon9_launches.csv", index=False)
print("Falcon 9 rows:", len(df))
display(df.head())
display(df.isna().sum().sort_values(ascending=False).head(12))

In [ ]:
# Landing target: only launches with a recorded landing attempt and a known landing result are labelled.
labelled = df[df["landing_attempt"].eq(True) & df["landing_success"].notna()].copy()
labelled["Class"] = labelled["landing_success"].astype(int)
labelled.to_csv(PROC / "spacex_falcon9_landing_model.csv", index=False)
print("Labelled landing attempts:", len(labelled))
print("Landing success rate (%):", round(100 * labelled["Class"].mean(), 2) if len(labelled) else "No labelled rows")
labelled["Class"].value_counts(dropna=False)

## Important interpretation
The API includes current historical launch records, so this dataset may differ from the older course snapshot used in IBM's graded labs. Keep the source and retrieval date documented. Do not label a launch as a landing failure merely because `landing_success` is missing; unknown outcomes are excluded from the labelled modelling dataset.